# Proyecto Centinela - Fase 1: Centinela de la Papa
### Redes Neuronales - Deep Learning (EA-N-F-004) - Maestria en Ciencia de Datos, USTA
Integrantes: Juan Guillermo Montano Junca y Jair Andres Valderrama Cano - version v1

Linea base con un MLP sobre la imagen aplanada a tabla (rasgos de color y textura por hoja) para predecir **hoja enferma: si/no** (tizon temprano o tardio frente a sana). Datos: Irish Potato Imagery Dataset (Zenodo, CC BY 4.0). El clima de NASA POWER entra en la Fase 2.

Estructura: 0 entorno - 1 datos - 2 EDA y particion - 3 arquitectura y capacidad - 4 bucle - 5 diagnostico y lineas base - 6 regularizacion - 7 umbral por costo, prueba, errores y etica - 8 autoevaluacion - anexos.

## 0. Preparacion del entorno

In [ ]:
import io, zipfile, os, random, urllib.request
import numpy as np, pandas as pd
import torch, torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, classification_report, roc_auc_score,
                             precision_recall_curve, recall_score, precision_score, f1_score)

SEMILLA = 42
random.seed(SEMILLA); np.random.seed(SEMILLA); torch.manual_seed(SEMILLA)   # semilla en las tres librerias
print("torch", torch.__version__, "| GPU:", torch.cuda.is_available())

## 1. El problema y los datos
Cada foto de hoja se resume en rasgos tabulares de color y textura: es la "version aplanada" de la imagen. En la Fase 2 se reemplaza por una CNN.

Todo corre en la nube de Colab, sin Drive: los zips se bajan al disco temporal (`/content`), se procesan de a uno y se borran. Solo queda `rasgos_papa.csv` (pocos MB). `POR_CLASE = 1500` para iterar; `None` para todo.

In [ ]:
ZIPS = {   # clase -> (nombre del zip, URL de Zenodo, etiqueta 0=sana 1=enferma)
    "sana":     ("healthy.zip",  "https://zenodo.org/records/8286529/files/healthy.zip?download=1",  0),
    "temprano": ("earlyblt.zip", "https://zenodo.org/records/8286529/files/earlyblt.zip?download=1", 1),
    "tardio":   ("lateblt.zip",  "https://zenodo.org/records/8286529/files/lateblt.zip?download=1",  1),
}
POR_CLASE   = 1500
LADO        = 128
CSV         = "rasgos_papa.csv"
BORRAR_ZIPS = True

In [ ]:
def rasgos_imagen(img, lado=LADO):
    """Resume una hoja en rasgos de color y textura."""
    a = np.asarray(img.convert("RGB").resize((lado, lado)), dtype=np.float32)
    R, G, B = a[:, :, 0], a[:, :, 1], a[:, :, 2]
    gris = 0.299 * R + 0.587 * G + 0.114 * B
    exg = 2 * G - R - B                              # exceso de verde: hoja sana = mas verde
    necrosis = float(np.mean((R > G) & (R > B)))     # fraccion parduzca/rojiza: lesiones
    gx, gy = np.gradient(gris)
    textura = float(np.std(np.sqrt(gx ** 2 + gy ** 2)))
    return [R.mean(), G.mean(), B.mean(), R.std(), G.std(), B.std(),
            float(exg.mean()), necrosis, float(gris.mean()), textura]

COLUMNAS = ["R_media","G_media","B_media","R_std","G_std","B_std",
            "exg_media","necrosis_frac","brillo_media","textura"]

def descargar(nombre, url):
    if not os.path.exists(nombre):
        print("Descargando", nombre, "..."); urllib.request.urlretrieve(url, nombre)

def construir_tabla():
    filas, y, clases = [], [], []
    for clase, (nombre, url, etiqueta) in ZIPS.items():
        descargar(nombre, url)
        with zipfile.ZipFile(nombre) as z:
            nombres = [n for n in z.namelist() if n.lower().endswith((".jpg",".jpeg",".png"))]
            random.Random(SEMILLA).shuffle(nombres)
            if POR_CLASE: nombres = nombres[:POR_CLASE]
            for n in nombres:
                try:
                    filas.append(rasgos_imagen(Image.open(io.BytesIO(z.read(n)))))
                    y.append(etiqueta); clases.append(clase)
                except Exception:
                    continue
        print(f"{clase}: {len(nombres)} fotos procesadas")
        if BORRAR_ZIPS: os.remove(nombre)
    df = pd.DataFrame(filas, columns=COLUMNAS); df["enferma"] = y; df["clase"] = clases
    return df

if os.path.exists(CSV):
    df = pd.read_csv(CSV); print("CSV cargado:", df.shape)
else:
    df = construir_tabla(); df.to_csv(CSV, index=False); print("Tabla construida:", df.shape)
df.head()

## 2. Exploracion (EDA): cada hallazgo decide algo

In [ ]:
print("Balance (enferma=1):"); print(df["enferma"].value_counts(normalize=True).round(3))
print("\nPor clase original:"); print(df["clase"].value_counts())
print("\nFaltantes por columna:", int(df[COLUMNAS].isna().sum().sum()))
df[COLUMNAS].describe().round(2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for v, eje in zip(["exg_media", "necrosis_frac"], ax):
    for et, col in [(0, "tab:green"), (1, "tab:brown")]:
        eje.hist(df[df.enferma == et][v], bins=30, alpha=0.6, color=col, label=("sana" if et == 0 else "enferma"))
    eje.set_title(v); eje.set_xlabel(v); eje.set_ylabel("fotos"); eje.legend()
plt.tight_layout(); plt.show()

### Decisiones que deja el EDA
| Hallazgo | Decision |
|---|---|
| Desbalance (mayoria "enferma", ~65/35) | particion **estratificada**; metricas por clase, no solo exactitud |
| Escalas muy distintas (0-255 frente a 0-1) | **StandardScaler** ajustado solo con train (anti-fuga) |
| "Enferma" agrupa temprano y tardio | se guarda la clase original para medir errores **por subgrupo** |
| `necrosis_frac` separa bien las clases | se usa como **escalar crudo** de linea base |
| Sin finca y solo la mitad con fecha (EXIF) | no se puede partir por grupo: se declara como limitacion (riesgo de casi duplicados) |

In [ ]:
idx = np.arange(len(df))
i_tr, i_tmp = train_test_split(idx, test_size=0.30, stratify=df["enferma"], random_state=SEMILLA)
i_va, i_te = train_test_split(i_tmp, test_size=0.50, stratify=df["enferma"].iloc[i_tmp], random_state=SEMILLA)

X = df[COLUMNAS].to_numpy("float32"); y = df["enferma"].to_numpy("float32")
escalador = StandardScaler().fit(X[i_tr])                     # SOLO train
X_tr, X_va, X_te = (escalador.transform(X[i]) for i in (i_tr, i_va, i_te))
y_tr, y_va, y_te = y[i_tr], y[i_va], y[i_te]
clase_te = df["clase"].to_numpy()[i_te]
print("train/val/test:", len(i_tr), len(i_va), len(i_te))

## 3. Arquitectura del MLP y capacidad
- **Salida en logit** (`nn.Linear(k, 1)`, sin sigmoide en el forward) con `BCEWithLogitsLoss` y etiqueta `(N, 1)` flotante. Poner `nn.Sigmoid()` en el modelo con esa perdida seria la doble sigmoide.
- **Por que ReLU en las ocultas:** la frontera sana/enferma no es una recta en el espacio de rasgos (por ejemplo, una hoja muy oscura puede tener poco verde sin estar enferma). Sin no linealidad, apilar capas equivale a una sola transformacion lineal y la red no podria doblar esa frontera. ReLU aporta esa no linealidad con un costo minimo y gradiente 1 en la zona activa.
- **Capacidad:** no se elige a ojo; se compara con medicion (celda siguiente).

In [ ]:
class CentinelaMLP(nn.Module):
    def __init__(self, d_entrada, oculta1=16, oculta2=8, p_dropout=0.0):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(d_entrada, oculta1), nn.ReLU(), nn.Dropout(p_dropout),   # oculta 1
            nn.Linear(oculta1, oculta2),  nn.ReLU(), nn.Dropout(p_dropout),   # oculta 2
            nn.Linear(oculta2, 1))                                            # salida: LOGIT
    def forward(self, x):
        return self.red(x)

## 4. Perdida, optimizador y bucle de entrenamiento explicito

In [ ]:
def a_tensor(Xa, ya):
    return (torch.as_tensor(Xa, dtype=torch.float32),
            torch.as_tensor(ya, dtype=torch.float32).view(-1, 1))      # etiqueta (N,1) flotante

def entrenar(modelo, Xtr, ytr, Xva, yva, epocas=80, lr=1e-2, paciencia=15, weight_decay=0.0):
    Xtr, ytr = a_tensor(Xtr, ytr); Xva, yva = a_tensor(Xva, yva)
    criterio = nn.BCEWithLogitsLoss()
    optim = torch.optim.Adam(modelo.parameters(), lr=lr, weight_decay=weight_decay)
    hist = {"train": [], "val": []}
    mejor, mejor_ep, espera, mejor_estado = float("inf"), 0, 0, None
    for ep in range(1, epocas + 1):
        modelo.train()                      # Dropout activo
        logits = modelo(Xtr)                # 1 forward (logit)
        perdida = criterio(logits, ytr)     # 2 perdida
        optim.zero_grad()                   # 3 limpiar gradientes
        perdida.backward()                  # 4 backward
        optim.step()                        # 5 actualizar pesos
        modelo.eval()
        with torch.no_grad():
            pv = criterio(modelo(Xva), yva).item()
        hist["train"].append(perdida.item()); hist["val"].append(pv)
        if pv < mejor:
            mejor, mejor_ep, espera = pv, ep, 0
            mejor_estado = {k: v.clone() for k, v in modelo.state_dict().items()}   # guarda el mejor
        else:
            espera += 1
            if paciencia and espera >= paciencia:
                break
    modelo.load_state_dict(mejor_estado)    # restaura el minimo de validacion
    return hist, mejor_ep

def probas(modelo, Xa):
    modelo.eval()
    with torch.no_grad():
        return torch.sigmoid(modelo(torch.as_tensor(Xa, dtype=torch.float32))).numpy().ravel()

In [ ]:
# Comparacion medida de capacidad: tres tamanos, misma semilla y mismos datos
filas = []
for h1, h2 in [(4, 2), (16, 8), (64, 32)]:
    torch.manual_seed(SEMILLA)
    m = CentinelaMLP(len(COLUMNAS), h1, h2)
    h, ep = entrenar(m, X_tr, y_tr, X_va, y_va)
    filas.append({"capas": f"{h1}-{h2}", "params": sum(p.numel() for p in m.parameters()),
                  "val_min": round(min(h["val"]), 4), "epoca_min": ep,
                  "AUC_val": round(roc_auc_score(y_va, probas(m, X_va)), 4)})
tabla_cap = pd.DataFrame(filas); tabla_cap

Lectura: se elige el tamano con menor perdida de validacion; si dos quedan casi iguales se prefiere el mas pequeno (menos parametros, menos riesgo de sobreajuste). **Escribir aqui la decision con las cifras reales de la tabla.**

## 5. Diagnostico y lineas base comprometidas

In [ ]:
def plot_curvas(hist, titulo):
    plt.figure(figsize=(6, 3.5))
    plt.plot(range(1, len(hist["train"]) + 1), hist["train"], label="entrenamiento")
    plt.plot(range(1, len(hist["val"]) + 1), hist["val"], label="validacion")
    plt.xlabel("epoca"); plt.ylabel("perdida BCE"); plt.title(titulo); plt.legend(); plt.show()

def plot_matriz(y_true, y_pred, titulo="Matriz de confusion (prueba)"):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    plt.figure(figsize=(3.8, 3.3)); plt.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            plt.text(j, i, cm[i, j], ha="center", va="center")
    plt.xticks([0, 1], ["sana", "enferma"]); plt.yticks([0, 1], ["sana", "enferma"])
    plt.xlabel("prediccion"); plt.ylabel("real"); plt.title(titulo); plt.show()

# Linea base 1: clase mayoritaria (todo "enferma")
pred_may = np.ones_like(y_te)
print("Clase mayoritaria  -> exactitud:", round(float((pred_may == y_te).mean()), 3),
      "| recall enferma:", round(recall_score(y_te, pred_may), 3),
      "| recall sana: 0.0")

# Linea base 2: escalar crudo (necrosis_frac sin escalar), umbral elegido en VALIDACION
nec = df["necrosis_frac"].to_numpy()
cands = np.quantile(nec[i_va], np.linspace(0.02, 0.98, 97))
u_nec = max(cands, key=lambda u: f1_score(y_va, (nec[i_va] >= u).astype(int)))
pred_nec = (nec[i_te] >= u_nec).astype(int)
print("Escalar crudo      -> umbral", round(float(u_nec), 4),
      "| exactitud:", round(float((pred_nec == y_te).mean()), 3),
      "| recall enferma:", round(recall_score(y_te, pred_nec), 3),
      "| AUC:", round(roc_auc_score(y_te, nec[i_te]), 3))

## 6. Regularizacion: inducir y corregir el sobreajuste
Se induce el sobreajuste con una red grande, muchas epocas y sin freno, y se compara con tres intervenciones (Dropout, L2 y ambas), todas con la misma semilla. En cada corrida se nombra la epoca del minimo de validacion y la brecha entre entrenamiento y validacion.

In [ ]:
corridas = {"sin regularizar": dict(p=0.0, wd=0.0),
            "Dropout 0.3":     dict(p=0.3, wd=0.0),
            "L2 1e-3":         dict(p=0.0, wd=1e-3),
            "Dropout + L2":    dict(p=0.3, wd=1e-3)}
res, modelos, hists = [], {}, {}
for nombre, c in corridas.items():
    torch.manual_seed(SEMILLA)
    m = CentinelaMLP(len(COLUMNAS), 128, 64, p_dropout=c["p"])
    h, ep = entrenar(m, X_tr, y_tr, X_va, y_va, epocas=200, paciencia=None, weight_decay=c["wd"])
    modelos[nombre], hists[nombre] = m, h
    res.append({"corrida": nombre, "epoca_min_val": ep, "val_min": round(min(h["val"]), 4),
                "val_final": round(h["val"][-1], 4),
                "brecha_en_min": round(h["val"][ep - 1] - h["train"][ep - 1], 4)})
tabla_reg = pd.DataFrame(res); tabla_reg

In [ ]:
for nombre in ["sin regularizar", "Dropout + L2"]:
    plot_curvas(hists[nombre], f"{nombre} (minimo de validacion en la epoca {tabla_reg.set_index('corrida').loc[nombre,'epoca_min_val']})")

Lectura: si en "sin regularizar" la validacion toca un minimo y luego sube mientras el entrenamiento sigue bajando, eso es sobreajuste; la diferencia `val_final - val_min` lo cuantifica. El modelo final es el de la corrida con menor `val_min`, restaurado en su epoca minima (parada temprana: se guarda y se restaura el mejor estado). Si el sobreajuste no aparece pese a la capacidad, se documenta que se aumento y que se observo. **Escribir aqui la eleccion con las cifras reales.**

## 7. Umbral por costo, evaluacion en prueba, errores y etica
**Costo de los errores para el cliente.** Un falso negativo (hoja enferma marcada sana) deja avanzar el tizon, que puede acabar con un lote en dias. Un falso positivo implica una visita o fumigacion de mas. Fijamos que el falso negativo cuesta **5 veces** lo que un falso positivo y elegimos el umbral que minimiza el costo esperado **en validacion** (nunca en prueba).

In [ ]:
nombre_final = tabla_reg.sort_values("val_min").iloc[0]["corrida"]
modelo = modelos[nombre_final]
print("Modelo final:", nombre_final)
prob_va, prob_te = probas(modelo, X_va), probas(modelo, X_te)

C_FN, C_FP = 5, 1
def costo(y_true, prob, u):
    p = (prob >= u).astype(int)
    fn = int(((p == 0) & (y_true == 1)).sum()); fp = int(((p == 1) & (y_true == 0)).sum())
    return C_FN * fn + C_FP * fp
umbrales = np.linspace(0.05, 0.95, 91)
umbral = float(min(umbrales, key=lambda u: costo(y_va, prob_va, u)))
print("Umbral por costo (validacion):", round(umbral, 2), "| frente a 0.5 por defecto")

pred_te = (prob_te >= umbral).astype(int)          # una sola evaluacion en prueba
print("\nAUC prueba:", round(roc_auc_score(y_te, prob_te), 3))
plot_matriz(y_te, pred_te)
print(classification_report(y_te, pred_te, target_names=["sana", "enferma"], digits=3))

In [ ]:
# Errores por subgrupo: el recall de "enferma" se mira por tipo de tizon
sub = pd.DataFrame({"clase": clase_te, "real": y_te, "pred": pred_te})
por_sub = sub.groupby("clase").apply(lambda g: pd.Series({
    "n": len(g), "acierto": round(float((g.real == g.pred).mean()), 3)}))
por_sub

### Lectura tecnica y etica de los errores
- Frente a las lineas base: el MLP debe superar la clase mayoritaria en recall de "sana" (ella da 0) y al escalar crudo en AUC. **Escribir aqui la comparacion con las cifras reales.**
- Por subgrupo: si el tizon temprano (lesiones pequenas) tiene mucho menos acierto que el tardio, el sistema avisa tarde justo en la etapa en que mas sirve avisar. Eso se reporta como limitacion y orienta la Fase 2 (la CNN ve la lesion, los promedios de color la diluyen).
- **Principio del Marco Etico para la IA en Colombia: control humano de las decisiones.** Traduccion concreta para el cliente: la alerta **no dispara una fumigacion automatica**; recomienda una inspeccion del tecnico de extension al lote, que confirma antes de aplicar. Por eso el umbral favorece el recall: es preferible una inspeccion de mas que un brote sin aviso.
- **Privacidad:** se borran los metadatos EXIF (fecha y, en pocas fotos, GPS) antes de subir imagenes a la nube o a un asistente de IA.
- **Limite de transferencia:** el modelo se entreno con una sola variedad (Shangi) de Mbeya; no se debe usar en Cundinamarca sin recalibrar.

## 8. Autoevaluacion contra la rubrica (escala 1.0 a 5.0)
Bandas: Estrategico 4.6-5.0 / Autonomo 4.0-4.5 / Basico 3.0-3.9 / Inicial 1.0-2.9. Se aprueba con 3.0 o mas.

| Criterio (peso) | Evidencia en este cuaderno para nivel Estrategico |
|---|---|
| 1. Propuesta y EDA (15%) | tabla hallazgo -> decision; lineas base calculadas (clase mayoritaria y escalar crudo) |
| 2. MLP (30%) | logit + BCEWithLogitsLoss + etiqueta (N,1); ReLU justificada; capacidad elegida por comparacion medida |
| 3. Regularizacion (15%) | sobreajuste inducido; cuatro corridas en tabla; mejor estado guardado y restaurado |
| 4. Evaluacion y etica (15%) | umbral por costo en validacion; metricas por clase; errores por subgrupo; principio etico -> decision |
| 5. Uso de IA (15%) | bitacora con casos reales y auditoria (Anexo A y C) |
| 6. Comunicacion (10%) | informe aparte (PDF, 5 secciones, APA 7) con cifras que salen de este cuaderno |

## Anexo A - Auditoria a la IA
**Caso real del proyecto (principal).** Al redactar la Propuesta de Datos, el asistente de IA (Claude) afirmo que las fotos venian de cuatro regiones (Mbeya, Iringa, Njombe y Songwe) y armo la clave de alineacion y el anexo de acoplamiento sobre esas cuatro. La ficha de Zenodo dice que **todas** las fotos son de Mbeya. Evidencia de origen: la version v1 de la propuesta en el historial del repositorio y la retroalimentacion del docente en la compuerta. Correccion: se verifico la ficha, se reviso el EXIF de una muestra (19 de 40 con fecha, 1 de 40 con GPS) y la clave paso a (Mbeya, dia), con 10 filas reales. Fundamento: acoplar clima de regiones donde no hay fotos habria sido un acoplamiento artificial, es decir, fuga de datos.

**Caso inducido y declarado (complementario): doble sigmoide.** Se reproduce con la misma semilla y los mismos datos que el modelo correcto, y se mide con AUC y recall.

In [ ]:
class MLPDobleSigmoide(nn.Module):
    """Version rota: sigmoide en el forward + BCEWithLogitsLoss (sigmoide aplicada dos veces)."""
    def __init__(self, d):
        super().__init__()
        self.red = nn.Sequential(nn.Linear(d, 16), nn.ReLU(), nn.Linear(16, 8), nn.ReLU(),
                                 nn.Linear(8, 1), nn.Sigmoid())          # <- sigmoide de mas
    def forward(self, x): return self.red(x)

torch.manual_seed(SEMILLA)
m_rota = MLPDobleSigmoide(len(COLUMNAS))
h_rota, _ = entrenar(m_rota, X_tr, y_tr, X_va, y_va, paciencia=None)
torch.manual_seed(SEMILLA)
m_ok = CentinelaMLP(len(COLUMNAS))
h_ok, _ = entrenar(m_ok, X_tr, y_tr, X_va, y_va, paciencia=None)

for nombre, m, h in [("ROTA (doble sigmoide)", m_rota, h_rota), ("CORRECTA (logit)", m_ok, h_ok)]:
    p = probas(m, X_te)
    print(f"{nombre:<22} val_min={min(h['val']):.3f}  AUC={roc_auc_score(y_te, p):.3f}  "
          f"recall_enferma@0.5={recall_score(y_te, (p >= 0.5).astype(int)):.3f}  "
          f"prob_min={p.min():.2f} prob_max={p.max():.2f}")

Lectura: con la doble sigmoide la perdida no puede bajar de un piso y todas las probabilidades quedan por encima de 0.5, asi que con el umbral 0.5 todo sale "enferma" (recall 1.0 trivial), aunque el AUC casi se conserve porque el orden de los casos no cambia. Por eso las curvas dejan de servir para diagnosticar.

## Anexo B - Propuesta de Datos
Version aprobada en la compuerta: `propuesta/propuesta-de-datos.md` y su PDF, en el repositorio. Se adjunta con su fecha de aprobacion.

## Anexo C - Bitacora de IA
Cada entrada remite a la celda o seccion que afecta. **Revisar fechas, completar con los casos propios y firmar cada integrante antes de entregar.**

| Fecha | Herramienta | Sugerencia de la IA | Decision | Verificacion y evidencia | Afecta |
|---|---|---|---|---|---|
| 2026-10-30 | Claude | Clave de alineacion con 4 regiones de Tanzania | Modificada | Ficha de Zenodo: todas en Mbeya; senalado por el docente | Propuesta 5, Anexo A |
| 2026-10-30 | Claude | Describir NASA POWER como estacion meteorologica | Modificada | Es producto satelital y de reanalisis; senalado por el docente | Propuesta 2 |
| 2026-11-02 | Claude | Script para leer EXIF de una muestra | Aceptada | Se corrio en local: 19/40 con fecha, 1/40 con GPS | Propuesta 5, `revisar_exif.py` |
| 2026-11-02 | Claude | Cruzar fecha EXIF con clima diario de NASA POWER | Aceptada | Se corrio y produjo 10 filas reales | Anexo A, `anexo_a.csv` |
| 2026-11-02 | Claude | Estructura de este cuaderno (MLP, bucle, diagnostico) | Modificada | Se ejecuto completo; se revisaron salida en logit y orden del bucle | Secciones 3 a 7 |

| Integrante | Firma | Fecha |
|---|---|---|
| Juan Guillermo Montano Junca | | |
| Jair Andres Valderrama Cano | | |

## Anexo D - Informe tecnico-etico
PDF aparte, maximo 4 paginas sin referencias: 1) problema y cliente; 2) datos; 3) modelo y entrenamiento; 4) resultados y errores (FN y FP); 5) decision recomendada y riesgos eticos. APA 7. Cada cifra del informe debe salir de las salidas de este cuaderno.